# Liu2024 MOABB EEG + clinical metadata integration

This notebook tests the intended data flow: EEG is loaded through the existing MOABB/Braindecode project pipeline, while patient paralysis metadata comes from `data/moabb/MNE-liu2024-data/files/participants.tsv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root')

repo_root = find_repo_root(Path.cwd().resolve())
participants_path = repo_root / 'data/moabb/MNE-liu2024-data/files/participants.tsv'
assert participants_path.exists(), participants_path
participants_path

PosixPath('/workspaces/BRAINDECODE/data/moabb/MNE-liu2024-data/files/participants.tsv')

## 1. Validate the clinical metadata and MOABB subject IDs

In [2]:
participants = pd.read_csv(participants_path, sep='\t')
participants['subject'] = participants['Participant_ID'].str.extract(r'^sub-(\d+)$', expand=False).astype(int)
clinical_columns = ['Gender', 'Age', 'Duration', 'ParalysisSide', 'Handedness', 'IsFirstTime', 'StrokeLocation', 'NIHSS', 'MBI', 'mRS']
assert len(participants) == 50
assert participants['subject'].is_unique
assert set(participants['subject']) == set(range(1, 51))
assert participants[clinical_columns].notna().all().all()
assert set(participants['ParalysisSide']) == {'left', 'right'}
display(participants.head())
display(participants['ParalysisSide'].value_counts().rename('n_subjects').to_frame())
print('PASS: all 50 MOABB subjects have complete clinical records')

,Participant_ID,Gender,Age,Duration,ParalysisSide,Handedness,IsFirstTime,StrokeLocation,NIHSS,MBI,mRS,subject
0,sub-01,male,45,1,right,right,yes,Left pons,11,50,4,1
1,sub-02,male,60,2,left,right,yes,Right pons,3,55,4,2
2,sub-03,male,60,2,left,right,no,"Left cerebellum, bilateral paraventricular, Ri...",3,90,1,3
3,sub-04,male,56,14,right,right,yes,"Left frontal parietal cortex, Left centrum sem...",6,90,3,4
4,sub-05,female,44,4,left,right,yes,Left pons,4,60,4,5


,n_subjects
ParalysisSide,
left,28
right,22


PASS: all 50 MOABB subjects have complete clinical records


The downloaded subject-level table contains 28 left-side and 22 right-side paralysis records. This differs from MOABB's aggregate descriptive count, so the table is treated as the operational subject mapping.

## 2. Load and window MOABB Liu2024 through reusable project code

Only subject 1 is loaded so this audit remains quick. The training scripts use the same loader and preprocessing functions.

In [3]:
from eeg_bci.data.moabb import build_moabb_dataset

dataset_cfg = OmegaConf.load(repo_root / 'configs/dataset/liu2024.yaml')
preprocessing_cfg = OmegaConf.load(repo_root / 'configs/preprocessing/liu2024.yaml')
dataset_cfg.subject_ids = [1]
windows, dataset_info = build_moabb_dataset(dataset_cfg, preprocessing_cfg)
metadata = windows.get_metadata().reset_index(drop=True)
sample_x, sample_y, *_ = windows[0]
assert (dataset_info.n_chans, dataset_info.n_outputs, dataset_info.n_times) == (29, 2, 2000)
assert tuple(sample_x.shape) == (29, 2000)
assert len(windows) == 40
assert metadata['target'].value_counts().to_dict() == {0: 20, 1: 20}
print(f'PASS: model input={tuple(sample_x.shape)}, windows={len(windows)}')
display(metadata.head())

Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:78: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(


PASS: model input=(29, 2000), windows=40


,i_window_in_trial,i_start_in_trial,i_stop_in_trial,target,subject,session,run
0,0,1003,3003,0,1,0,0
1,0,5002,7002,1,1,0,0
2,0,9002,11002,0,1,0,0
3,0,13003,15003,1,1,0,0
4,0,17003,19003,0,1,0,0


## 3. Join every EEG window to clinical metadata

In [4]:
clinical_by_subject = participants.set_index('subject')
window_clinical = metadata.join(clinical_by_subject[['ParalysisSide', 'Handedness', 'NIHSS', 'MBI', 'mRS']], on='subject', validate='many_to_one')
assert window_clinical['ParalysisSide'].notna().all()
assert set(window_clinical['subject']) == {1}
display(window_clinical.head())
print('PASS: every EEG window has matching clinical metadata')

,i_window_in_trial,i_start_in_trial,i_stop_in_trial,target,subject,session,run,ParalysisSide,Handedness,NIHSS,MBI,mRS
0,0,1003,3003,0,1,0,0,right,right,11,50,4
1,0,5002,7002,1,1,0,0,right,right,11,50,4
2,0,9002,11002,0,1,0,0,right,right,11,50,4
3,0,13003,15003,1,1,0,0,right,right,11,50,4
4,0,17003,19003,0,1,0,0,right,right,11,50,4


PASS: every EEG window has matching clinical metadata


## 4. Convert left/right targets to affected/unaffected

In [5]:
LEFT_HAND, RIGHT_HAND = 0, 1
AFFECTED, UNAFFECTED = 0, 1

def to_affected_unaffected(hand_target: int, paralysis_side: str) -> int:
    if hand_target not in {LEFT_HAND, RIGHT_HAND}:
        raise ValueError(f'Expected target 0 or 1, got {hand_target}')
    if paralysis_side == 'left':
        return AFFECTED if hand_target == LEFT_HAND else UNAFFECTED
    if paralysis_side == 'right':
        return AFFECTED if hand_target == RIGHT_HAND else UNAFFECTED
    raise ValueError(f'Unsupported paralysis side: {paralysis_side!r}')

window_clinical['canonical_target'] = [to_affected_unaffected(int(target), side) for target, side in zip(window_clinical['target'], window_clinical['ParalysisSide'])]
window_clinical['canonical_label'] = window_clinical['canonical_target'].map({0: 'affected', 1: 'unaffected'})
assert window_clinical['canonical_target'].value_counts().to_dict() == {0: 20, 1: 20}
display(window_clinical[['subject', 'target', 'ParalysisSide', 'canonical_target', 'canonical_label']].head(6))

,subject,target,ParalysisSide,canonical_target,canonical_label
0,1,0,right,1,unaffected
1,1,1,right,0,affected
2,1,0,right,1,unaffected
3,1,1,right,0,affected
4,1,0,right,1,unaffected
5,1,1,right,0,affected


## 5. Test a leakage-safe chronological target split

For SPPM, the early 80% of the held-out subject can be unlabeled target-adaptation data and the final 20% remains untouched test data.

In [6]:
from eeg_bci.data.splitting.sources import _chronological_split_indices

adapt_idx, test_idx = _chronological_split_indices(windows, test_size=0.2, stratify=True)
assert len(adapt_idx) == 32 and len(test_idx) == 8
assert set(adapt_idx).isdisjoint(set(test_idx))
assert set(adapt_idx) | set(test_idx) == set(range(40))
assert np.bincount(metadata.iloc[adapt_idx]['target'], minlength=2).tolist() == [16, 16]
assert np.bincount(metadata.iloc[test_idx]['target'], minlength=2).tolist() == [4, 4]
assert metadata.iloc[adapt_idx]['i_start_in_trial'].max() < metadata.iloc[test_idx]['i_start_in_trial'].min()
split_summary = pd.DataFrame({'partition': ['target_adaptation', 'target_test'], 'n_windows': [32, 8], 'left_hand': [16, 4], 'right_hand': [16, 4]})
display(split_summary)
print('PASS: chronological split is 32 adaptation / 8 test')

,partition,n_windows,left_hand,right_hand
0,target_adaptation,32,16,16
1,target_test,8,4,4


PASS: chronological split is 32 adaptation / 8 test


## Conclusion

Use MOABB/Braindecode for EEG, join the cache-local `participants.tsv` by subject ID, convert labels only for affected/unaffected experiments, and reserve the final chronological 20% for target evaluation. Clinical metadata is not a replacement EEG dataset.

# dataset/dataloader for CFSPMNET

In [8]:
import torch
from torch.utils.data import DataLoader, Dataset

# Notebook-only LOSO prototype: subjects 1-2 are labeled sources; subject 3 is target.
loader_dataset_cfg = OmegaConf.load(repo_root / 'configs/dataset/liu2024.yaml')
loader_preprocessing_cfg = OmegaConf.load(repo_root / 'configs/preprocessing/liu2024.yaml')
loader_dataset_cfg.subject_ids = [1, 2, 3]
all_windows, all_dataset_info = build_moabb_dataset(loader_dataset_cfg, loader_preprocessing_cfg)
all_metadata = all_windows.get_metadata().reset_index(drop=True)
target_subject = 3
source_indices = np.flatnonzero(all_metadata['subject'].to_numpy() != target_subject)

def split_target_chronologically(metadata, subject_id, test_size=0.2):
    target_rows = np.flatnonzero(metadata['subject'].to_numpy() == subject_id)
    adapt, test = [], []
    for label in sorted(metadata.iloc[target_rows]['target'].unique()):
        rows = target_rows[metadata.iloc[target_rows]['target'].to_numpy() == label]
        ordered = rows[np.argsort(metadata.iloc[rows]['i_start_in_trial'].to_numpy())]
        n_test = max(1, int(np.ceil(len(ordered) * test_size)))
        adapt.extend(ordered[:-n_test]); test.extend(ordered[-n_test:])
    return np.asarray(adapt), np.asarray(test)

target_adapt_indices, target_test_indices = split_target_chronologically(all_metadata, target_subject)
assert (len(source_indices), len(target_adapt_indices), len(target_test_indices)) == (80, 32, 8)
assert set(source_indices).isdisjoint(target_adapt_indices)
assert set(source_indices).isdisjoint(target_test_indices)
assert set(target_adapt_indices).isdisjoint(target_test_indices)

paralysis_by_subject = participants.set_index('subject')['ParalysisSide'].to_dict()
def canonical_target(original_target, subject_id):
    return to_affected_unaffected(int(original_target), paralysis_by_subject[int(subject_id)])

class CanonicalLabeledDataset(Dataset):
    def __init__(self, windows, metadata, indices):
        self.windows, self.metadata = windows, metadata
        self.indices = np.asarray(indices, dtype=np.int64)
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        idx = int(self.indices[item]); x, original_y, *_ = self.windows[idx]
        subject = int(self.metadata.iloc[idx]['subject'])
        return torch.as_tensor(x, dtype=torch.float32), torch.tensor(canonical_target(original_y, subject), dtype=torch.long)

class TargetPseudoDataset(Dataset):
    def __init__(self, windows, indices, n_classes=2):
        self.windows = windows; self.indices = np.asarray(indices, dtype=np.int64)
        self.pseudo_labels = torch.zeros((len(indices), n_classes), dtype=torch.float32)
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        x, *_ = self.windows[int(self.indices[item])]
        return torch.as_tensor(x, dtype=torch.float32), self.pseudo_labels[item], torch.tensor(item)
    def update_pseudo_labels(self, local_indices, labels):
        self.pseudo_labels[torch.as_tensor(local_indices)] = torch.as_tensor(labels)
    def active_ratio(self): return float((self.pseudo_labels.sum(1) > 0).float().mean())

source_dataset = CanonicalLabeledDataset(all_windows, all_metadata, source_indices)
target_adaptation_dataset = TargetPseudoDataset(all_windows, target_adapt_indices)
target_test_dataset = CanonicalLabeledDataset(all_windows, all_metadata, target_test_indices)
kwargs = dict(batch_size=32, num_workers=0, pin_memory=torch.cuda.is_available())
source_loader = DataLoader(source_dataset, shuffle=True, **kwargs)
target_adaptation_loader = DataLoader(target_adaptation_dataset, shuffle=True, **kwargs)
target_initialization_loader = DataLoader(target_adaptation_dataset, shuffle=False, **kwargs)
target_test_loader = DataLoader(target_test_dataset, shuffle=False, **kwargs)
source_x, source_y = next(iter(source_loader))
target_x, target_pseudo, target_local_idx = next(iter(target_initialization_loader))
test_x, test_y = next(iter(target_test_loader))
assert source_x.shape == (32, 29, 2000) and source_y.shape == (32,)
assert target_x.shape == (32, 29, 2000) and target_pseudo.shape == (32, 2)
assert test_x.shape == (8, 29, 2000) and test_y.shape == (8,)
assert torch.all(target_pseudo == 0)
accepted = torch.tensor([0, 3, 7]); new_pseudo = torch.tensor([[1., 0.], [0., 1.], [1., 0.]])
target_adaptation_dataset.update_pseudo_labels(accepted, new_pseudo)
assert target_adaptation_dataset.active_ratio() == 3 / 32
loader_audit = pd.DataFrame({'loader':['source','target_adaptation','target_test'], 'samples':[len(source_dataset),len(target_adaptation_dataset),len(target_test_dataset)], 'first_batch_x_shape':[tuple(source_x.shape),tuple(target_x.shape),tuple(test_x.shape)], 'returned_target_shape':[tuple(source_y.shape),tuple(target_pseudo.shape),tuple(test_y.shape)], 'target_kind':['canonical_label','pseudo_distribution','canonical_label']})
display(loader_audit)
print('PASS: CFSPMNet dataloaders, shapes, isolation, and pseudo-label updates are valid')

/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:78: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(


,loader,samples,first_batch_x_shape,returned_target_shape,target_kind
0,source,80,"(32, 29, 2000)","(32,)",canonical_label
1,target_adaptation,32,"(32, 29, 2000)","(32, 2)",pseudo_distribution
2,target_test,8,"(8, 29, 2000)","(8,)",canonical_label


PASS: CFSPMNet dataloaders, shapes, isolation, and pseudo-label updates are valid
